In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from skrebate import ReliefF

df = pd.read_csv('dataset.csv', parse_dates=['datetime'])

train_start, train_end = '2024-01-01', '2024-03-30'
valid_start, valid_end = '2024-04-01', '2024-06-30'

treino = df[(df['datetime'] >= train_start) & (df['datetime'] <= train_end)].copy()
validacao = df[(df['datetime'] >= valid_start) & (df['datetime'] <= valid_end)].copy()

for sub_df in [treino, validacao]:
    sub_df.drop(columns=['datetime', 'date', 'close', 'open', 'low', 'high','volume', 'average', 'amount_stock', 'id_ticker', 'business', 'AD_Line'], 
                inplace=True, errors='ignore')

def remove_non_numeric(df):
    return df.select_dtypes(include=[np.number])

X_train = remove_non_numeric(treino.drop(columns=['trend']))
y_train = treino['trend']

X_valid = remove_non_numeric(validacao.drop(columns=['trend']))
y_valid = validacao['trend']

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_train)

In [ ]:
def cfs_subset_eval(X, y):
    corr_matrix = X.corr().abs()
    feature_target_corr = X.apply(lambda col: col.corr(y)).abs()
    selected = feature_target_corr.sort_values(ascending=False).index.tolist()
    return selected

def classifier_attribute_eval(X, y):
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X, y)
    importances = clf.feature_importances_
    return list(X.columns[np.argsort(importances)[::-1]])

def correlation_attribute_eval(X, y):
    corrs = X.apply(lambda col: abs(col.corr(y)))
    return list(corrs.sort_values(ascending=False).index)

def pca_ranking(X):
    pca = PCA(n_components=min(5, X.shape[1]))
    pca.fit(X)
    component_weights = np.abs(pca.components_[0])
    return list(X.columns[np.argsort(component_weights)[::-1]])

def information_gain_eval(X_scaled, y):
    info_gain = mutual_info_classif(X_scaled, y, random_state=42)
    info_gain_series = pd.Series(info_gain, index=X_train.columns)
    return list(info_gain_series.sort_values(ascending=False).index)

def reliefF_eval(X_scaled, y):
    relief = ReliefF(n_neighbors=100, n_features_to_select=X_train.shape[1])
    relief.fit(X_scaled, y)
    relief_scores = pd.Series(relief.feature_importances_, index=X_train.columns)
    return list(relief_scores.sort_values(ascending=False).index)

rankings = {
    "CFS_SubsetEval": cfs_subset_eval(X_train, y_train),
    "ClassifierAttributeEval": classifier_attribute_eval(X_train, y_train),
    "CorrelationAttributeEval": correlation_attribute_eval(X_train, y_train),
    "PCA": pca_ranking(X_train),
    "Information_Gain": information_gain_eval(X_scaled, y_train),
    "ReliefF": reliefF_eval(X_scaled, y_train)
}

for method, ranking in rankings.items():
    print(f"\n Top Features - {method}:")
    print(ranking[:23])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

df = pd.read_csv('normalizados_passo2.csv', parse_dates=['datetime'])

treino = df[(df['datetime'] >= '2024-01-01') & (df['datetime'] <= '2024-03-30')].copy()
validacao = df[(df['datetime'] >= '2024-04-01') & (df['datetime'] <= '2024-06-30')].copy()

drop_cols = ['datetime', 'date', 'close', 'open', 'low', 'high',
             'volume', 'average', 'amount_stock', 'id_ticker', 'business', 'AD_Line']
treino.drop(columns=drop_cols, inplace=True, errors='ignore')
validacao.drop(columns=drop_cols, inplace=True, errors='ignore')

X_train = treino.drop(columns=['trend'])
y_train = treino['trend']

X_valid = validacao.drop(columns=['trend'])
y_valid = validacao['trend']

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_valid_scaled = pd.DataFrame(scaler.transform(X_valid), columns=X_valid.columns)

In [ ]:
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_validate

feature_methods = {
    'CFS_SubsetEval':['EMA_3', 'EMA_5', 'EMA_7', 'EMA_9', 'EMA_11', 'SMA_3', 'SMA_5', 'SMA_9', 'SMA_7', 'SMA_11', 'Bollinger_Norm', 'std_open3', 'std_close3', 'std_open11', 'std_close7', 'std_open5', 'std_close11', 'std_open7', 'ADXR', 'std_close5', 'std_close9', 'std_open9'],
    'ClassifierAttributeEval': ['EMA_3', 'SMA_3', 'ADXR', 'Bollinger_Norm', 'EMA_5', 'SMA_5', 'std_close3', 'std_open3', 'std_close5', 'std_open5', 'std_open7', 'SMA_7', 'std_close7', 'SMA_9', 'std_open9', 'std_close11', 'EMA_11', 'std_close9', 'std_open11', 'SMA_11', 'EMA_7', 'EMA_9'],
    'CorrelationAttributeEval': ['EMA_3', 'EMA_5', 'EMA_7', 'EMA_9', 'EMA_11', 'SMA_3', 'SMA_5', 'SMA_9', 'SMA_7', 'SMA_11', 'Bollinger_Norm', 'std_open3', 'std_close3', 'std_open11', 'std_close7', 'std_open5', 'std_close11', 'std_open7', 'ADXR', 'std_close5', 'std_close9', 'std_open9'],
    'PCA':['Bollinger_Norm', 'SMA_11', 'EMA_11', 'SMA_9', 'EMA_9', 'SMA_7', 'EMA_7', 'EMA_5', 'SMA_5', 'EMA_3', 'SMA_3', 'std_open5', 'std_open7', 'std_open3', 'ADXR', 'std_close9', 'std_open11', 'std_close7', 'std_close5', 'std_close11', 'std_open9', 'std_close3'],
    'Information_Gain': ['SMA_3', 'EMA_3', 'EMA_5', 'Bollinger_Norm', 'SMA_9', 'std_open3', 'ADXR', 'SMA_11', 'EMA_11', 'SMA_5', 'SMA_7', 'std_close9', 'EMA_7', 'std_close5', 'EMA_9', 'std_open9', 'std_close7', 'std_close3', 'std_open5', 'std_open7', 'std_open11', 'std_close11'],
    'ReliefF': ['EMA_3', 'EMA_5', 'SMA_3', 'EMA_7', 'EMA_9', 'EMA_11', 'SMA_5', 'SMA_11', 'SMA_9', 'SMA_7', 'std_close3', 'std_open3', 'ADXR', 'Bollinger_Norm', 'std_close9', 'std_close11', 'std_open5', 'std_close5', 'std_close7', 'std_open9', 'std_open7', 'std_open11']
}

# Dicionários separados para armazenar as curvas de treino e de validação
results_train = {}
results_valid = {}

cv = StratifiedKFold(n_splits=9, shuffle=False, random_state=42)
#cv = TimeSeriesSplit(n_splits=9)

for method, features in feature_methods.items():
    acc_list_train = []
    acc_list_valid = []
    print(f"Avaliando: {method}")  
    
    for i in range(1, len(features)+1):
        selected_features = features[:i]
        X_tr = X_train_scaled[selected_features]
        
        model = RandomForestClassifier(n_estimators=100, random_state=42)   
        
        # cross_validate extrai as métricas de ambos os conjuntos das dobras
        scores = cross_validate(model, X_tr, y_train, cv=cv, scoring='accuracy', return_train_score=True)
        
        acc_list_train.append(np.mean(scores['train_score']))
        acc_list_valid.append(np.mean(scores['test_score']))

    results_train[method] = acc_list_train
    results_valid[method] = acc_list_valid

In [ ]:
import json

resultados_completos = {
    'train': results_train,
    'valid': results_valid
}

# Salva a estrutura completa contendo as métricas de treino e validação
with open('resultados_cv.json', 'w') as arquivo:
    json.dump(resultados_completos, arquivo, indent=4)

with open('resultados_cv.json', 'r') as arquivo:
    resultados_completos = json.load(arquivo)

In [ ]:
import matplotlib.pyplot as plt

for method, accs in resultados_completos['valid'].items():
    plt.figure(figsize=(8, 5))
    
    # Plota a linha de acurácia da validação
    plt.plot(range(1, len(accs) + 1), accs, color='b', label=method)

    # Caso já tenhas o modelo escolhido e quais pontos foram determinados, assim se marca eles 
    '''if method == 'ClassifierAttributeEval':
        plt.plot(8, accs[7], marker='o', color='green', markersize=8, 
                 linestyle='None', label='8 Features')
        plt.plot(18, accs[17], marker='o', color='red', markersize=8, 
                 linestyle='None', label='18 Features')'''
    
    # Configurações dos eixos
    plt.xlabel('Number of Features Used') 
    plt.ylabel('Accuracy in Validation')
    
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    
    # Define o nome do arquivo e salva em PDF
    safe_name = method.replace(' ', '_')
    filename = f"grafico_{safe_name}.pdf"
    
    plt.savefig(filename, format="pdf", bbox_inches='tight')
    
    plt.close()

print("Todos os gráficos foram exportados individualmente em PDF")